In [ ]:
# =========================================================
# UPDATED CONFIG + DATASET FOR NEW CSI-BENCH STRUCTURE
# Replace your current CONFIG through CSIBenchDataset block with this
# =========================================================

import os
import glob
import json
import h5py
import gc
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset

# =========================================================
# CONFIG
# =========================================================

TASK_NAME = "HumanActivityRecognition"

SEED = 42
EPOCHS = 30
PATIENCE = 8
MIN_DELTA = 1e-4

BATCH_SIZE = 8
ACCUM_STEPS = 4
LR = 8e-4
WEIGHT_DECAY = 1e-4

TARGET_SUBCARRIERS = 56
TARGET_TIME_LEN = 500

EMBED_DIM = 64
DEPTH = 2
NUM_HEADS = 2
MLP_RATIO = 2
DROPOUT = 0.1

FINE_PATCH = 8
FINE_STRIDE = 4
COARSE_PATCH = 32
COARSE_STRIDE = 16

USE_GRADIENT_CHECKPOINTING = True
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

ABLATION_DIR = "/kaggle/working/ablation_results"
os.makedirs(ABLATION_DIR, exist_ok=True)

# =========================================================
# AUTO-DETECT NEW CSI-BENCH ROOT
# =========================================================

def find_task_root(task_name="HumanActivityRecognition"):
    candidate_roots = []

    # Most likely new paths
    preferred_candidates = [
        f"/kaggle/input/datasets/guozhenjennzhu/csi-bench/Multitask/{task_name}",
        f"/kaggle/input/csi-bench/Multitask/{task_name}",
        f"/kaggle/input/**/Multitask/{task_name}",
        f"/kaggle/input/**/{task_name}",
    ]

    for pattern in preferred_candidates:
        for p in glob.glob(pattern, recursive=True):
            if os.path.isdir(p):
                candidate_roots.append(p)

    # Remove duplicates while preserving order
    candidate_roots = list(dict.fromkeys(candidate_roots))

    print("\nCandidate task roots:")
    for p in candidate_roots:
        print(" -", p)

    for p in candidate_roots:
        split_path = os.path.join(p, "splits", "train_id.json")
        meta_path = os.path.join(p, "metadata", "sample_metadata.csv")

        if os.path.exists(split_path) and os.path.exists(meta_path):
            print("\n✅ Using ROOT:")
            print(p)
            return p

    raise FileNotFoundError(
        f"Could not find {task_name} folder with both:\n"
        "  splits/train_id.json\n"
        "  metadata/sample_metadata.csv\n\n"
        "Run this to inspect paths:\n"
        "!find /kaggle/input -maxdepth 6 -type d -name '*HumanActivity*'"
    )

ROOT = find_task_root(TASK_NAME)

# Useful base paths for robust file resolution
CSI_BENCH_BASE = os.path.dirname(os.path.dirname(ROOT))  # usually .../csi-bench
MULTITASK_BASE = os.path.dirname(ROOT)                   # usually .../Multitask

print("\n========== ROOT VERIFICATION ==========")
print("ROOT:", ROOT)
print("Has splits:", os.path.exists(os.path.join(ROOT, "splits")))
print("Has metadata:", os.path.exists(os.path.join(ROOT, "metadata")))
print("Train split:", os.path.exists(os.path.join(ROOT, "splits", "train_id.json")))
print("Val split:", os.path.exists(os.path.join(ROOT, "splits", "val_id.json")))
print("Test split:", os.path.exists(os.path.join(ROOT, "splits", "test_id.json")))
print("Metadata CSV:", os.path.exists(os.path.join(ROOT, "metadata", "sample_metadata.csv")))

# =========================================================
# HELPERS
# =========================================================

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

set_seed(SEED)
cleanup()

# =========================================================
# SPLIT CHECK + LABEL MAPPING
# =========================================================

def load_split_ids(root_dir, split):
    split_file = os.path.join(root_dir, "splits", f"{split}_id.json")

    if not os.path.exists(split_file):
        return set()

    with open(split_file, "r") as f:
        return set(map(str, json.load(f)))

metadata_path = os.path.join(ROOT, "metadata", "sample_metadata.csv")
metadata = pd.read_csv(metadata_path)

metadata["id"] = metadata["id"].astype(str)
metadata["file_path"] = metadata["file_path"].astype(str)

train_ids = load_split_ids(ROOT, "train")
val_ids = load_split_ids(ROOT, "val")
test_ids = load_split_ids(ROOT, "test")

print("\n========== SPLIT CHECK ==========")
print("Train IDs:", len(train_ids))
print("Val IDs  :", len(val_ids))
print("Test IDs :", len(test_ids))
print("Train-Val overlap :", len(train_ids & val_ids))
print("Train-Test overlap:", len(train_ids & test_ids))
print("Val-Test overlap  :", len(val_ids & test_ids))

train_meta = metadata[metadata["id"].isin(train_ids)].copy()
train_labels = sorted(train_meta["label"].unique())

LABEL_MAPPING = {label: idx for idx, label in enumerate(train_labels)}
INV_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}

print("\nLabel mapping:")
print(LABEL_MAPPING)

# =========================================================
# UPDATED DATASET
# =========================================================

# =========================================================
# FIXED DATASET CLASS FOR NEW CSI-BENCH STRUCTURE
# Replace your current CSIBenchDataset class with this
# =========================================================

class CSIBenchDataset(Dataset):
    def __init__(
        self,
        root_dir,
        split="train",
        normalize=True,
        target_subcarriers=56,
        target_time_len=500,
        label_mapping=None
    ):
        self.root_dir = root_dir
        self.split = split
        self.normalize = normalize
        self.target_subcarriers = target_subcarriers
        self.target_time_len = target_time_len

        # Example:
        # root_dir        = .../csi-bench/Multitask/HumanActivityRecognition
        # multitask_base  = .../csi-bench/Multitask
        # csi_bench_base  = .../csi-bench
        self.multitask_base = os.path.dirname(root_dir)
        self.csi_bench_base = os.path.dirname(self.multitask_base)

        split_file = os.path.join(root_dir, "splits", f"{split}_id.json")
        metadata_path = os.path.join(root_dir, "metadata", "sample_metadata.csv")

        if not os.path.exists(split_file):
            raise FileNotFoundError(f"Split file not found: {split_file}")

        if not os.path.exists(metadata_path):
            raise FileNotFoundError(f"Metadata file not found: {metadata_path}")

        with open(split_file, "r") as f:
            self.sample_ids = list(map(str, json.load(f)))

        self.metadata = pd.read_csv(metadata_path)
        self.metadata["id"] = self.metadata["id"].astype(str)
        self.metadata["file_path"] = self.metadata["file_path"].astype(str)

        self.meta_dict = {
            str(row["id"]): row
            for _, row in self.metadata.iterrows()
        }

        if label_mapping is None:
            labels = sorted(self.metadata["label"].unique())
            self.label_mapping = {label: idx for idx, label in enumerate(labels)}
        else:
            self.label_mapping = label_mapping

        self.inv_label_mapping = {v: k for k, v in self.label_mapping.items()}

        print(f"Loaded {len(self.sample_ids)} samples for split: {split}")

    def __len__(self):
        return len(self.sample_ids)

    def resolve_file_path(self, raw_path):
        """
        Handles CSI-Bench metadata paths like:
        ../../sub_Human_h5/user_U01/...

        Important:
        Do NOT use raw.replace('./', '') because it corrupts ../../ paths.
        """
        raw = str(raw_path).replace("\\", "/").strip()

        # Remove only a leading "./", not every "./" occurrence
        if raw.startswith("./"):
            raw = raw[2:]

        candidates = []

        if os.path.isabs(raw):
            candidates.append(os.path.normpath(raw))

        # Normal relative-path candidates
        candidates.extend([
            os.path.normpath(os.path.join(self.root_dir, raw)),
            os.path.normpath(os.path.join(self.multitask_base, raw)),
            os.path.normpath(os.path.join(self.csi_bench_base, raw)),
            os.path.normpath(os.path.join("/kaggle/input", raw)),
        ])

        # Explicit handling for shared CSI storage folders in new structure
        shared_markers = [
            "sub_Human_h5/",
            "sub_Human_mat/",
            "HumanActivityRecognition/",
            "Multitask/HumanActivityRecognition/",
        ]

        for marker in shared_markers:
            if marker in raw:
                suffix = raw.split(marker, 1)[1]

                if marker.startswith("sub_Human_h5"):
                    candidates.extend([
                        os.path.normpath(os.path.join(self.multitask_base, "sub_Human_h5", suffix)),
                        os.path.normpath(os.path.join(self.csi_bench_base, "sub_Human_h5", suffix)),
                        os.path.normpath(os.path.join(self.root_dir, "sub_Human_h5", suffix)),
                    ])

                elif marker.startswith("sub_Human_mat"):
                    candidates.extend([
                        os.path.normpath(os.path.join(self.multitask_base, "sub_Human_mat", suffix)),
                        os.path.normpath(os.path.join(self.csi_bench_base, "sub_Human_mat", suffix)),
                        os.path.normpath(os.path.join(self.root_dir, "sub_Human_mat", suffix)),
                    ])

                else:
                    candidates.append(os.path.normpath(os.path.join(self.root_dir, suffix)))

        # Direct filename fallback using targeted search only if needed
        base = os.path.basename(raw)

        for path in candidates:
            if os.path.exists(path):
                return path

        # Last fallback: targeted search under Multitask, not full /kaggle/input
        matches = []
        for search_base in [self.multitask_base, self.csi_bench_base]:
            pattern = os.path.join(search_base, "**", base)
            matches.extend(glob.glob(pattern, recursive=True))

        if matches:
            return matches[0]

        raise FileNotFoundError(
            "Could not resolve CSI file path.\n"
            f"metadata file_path: {raw_path}\n\n"
            "Tried:\n" + "\n".join(candidates[:12]) + "\n\n"
            f"Also searched for filename: {base}"
        )

    def load_h5(self, path):
        with h5py.File(path, "r") as f:
            keys = list(f.keys())

            for key in ["csi", "data", "CSI", "amplitude"]:
                if key in keys:
                    return f[key][:]

            return f[keys[0]][:]

    def to_ckt(self, data):
        data = np.array(data)

        if np.iscomplexobj(data):
            data = np.abs(data)

        data = data.astype(np.float32)

        if data.ndim == 2:
            a, b = data.shape

            if a <= b:
                return data[np.newaxis, :, :]
            else:
                return data.T[np.newaxis, :, :]

        if data.ndim == 3:
            s0, s1, s2 = data.shape

            if s0 <= 8 and s1 <= self.target_subcarriers * 2:
                return data

            if s2 <= 8 and s0 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 0, 1))

            if s2 <= 8 and s1 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 1, 0))

            if s2 <= 16:
                return np.transpose(data, (2, 0, 1))

        raise ValueError(f"Unexpected CSI shape: {data.shape}")

    def standardize_subcarriers(self, x):
        C, K, T = x.shape

        if K > self.target_subcarriers:
            x = x[:, :self.target_subcarriers, :]
        elif K < self.target_subcarriers:
            pad_k = self.target_subcarriers - K
            pad = np.zeros((C, pad_k, T), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=1)

        return x

    def standardize_time(self, x):
        C, K, T = x.shape

        if T > self.target_time_len:
            x = x[:, :, :self.target_time_len]
        elif T < self.target_time_len:
            pad_t = self.target_time_len - T
            pad = np.zeros((C, K, pad_t), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=2)

        return x

    def __getitem__(self, idx):
        sample_id = str(self.sample_ids[idx])

        if sample_id not in self.meta_dict:
            raise KeyError(f"Sample ID not found in metadata: {sample_id}")

        meta = self.meta_dict[sample_id]

        file_path = self.resolve_file_path(meta["file_path"])
        csi = self.load_h5(file_path)

        x = self.to_ckt(csi)
        x = self.standardize_subcarriers(x)
        x = self.standardize_time(x)

        if self.normalize:
            x = (x - x.mean()) / (x.std() + 1e-6)

        label_name = meta["label"]

        if label_name not in self.label_mapping:
            raise KeyError(f"Label not found in label mapping: {label_name}")

        y = self.label_mapping[label_name]

        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)
# =========================================================
# CREATE DATASETS
# =========================================================

train_dataset = CSIBenchDataset(
    ROOT,
    split="train",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING
)

val_dataset = CSIBenchDataset(
    ROOT,
    split="val",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING
)

test_dataset = None
test_split_path = os.path.join(ROOT, "splits", "test_id.json")

if os.path.exists(test_split_path):
    test_dataset = CSIBenchDataset(
        ROOT,
        split="test",
        normalize=True,
        target_subcarriers=TARGET_SUBCARRIERS,
        target_time_len=TARGET_TIME_LEN,
        label_mapping=LABEL_MAPPING
    )

sample_x, sample_y = train_dataset[0]
CSI_CHANNELS = sample_x.shape[0]
NUM_CLASSES = len(LABEL_MAPPING)

print("\n========== DATA INFO ==========")
print("Sample shape:", sample_x.shape)
print("CSI channels:", CSI_CHANNELS)
print("Num classes:", NUM_CLASSES)
print("Example label:", sample_y)

In [ ]:
# ============================================================
# EdgeViT Ablation Models

# Properly formatted ablation code
# ============================================================

import os
import gc
import json
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.utils.checkpoint import checkpoint as torch_checkpoint

from sklearn.metrics import accuracy_score, f1_score


# ============================================================
# REQUIRED GLOBAL VARIABLES
# ============================================================
# This cell assumes the following variables already exist from
# your main training/data-loading cell:
#
# train_dataset
# val_dataset
# test_dataset
# LABEL_MAPPING
# CSI_CHANNELS
# TARGET_SUBCARRIERS
# TARGET_TIME_LEN
# NUM_CLASSES
# DEVICE
#
# Also define these hyperparameters before running:
# ============================================================

SEED = 42

EPOCHS = 30
PATIENCE = 8
MIN_DELTA = 1e-4

BATCH_SIZE = 8
ACCUM_STEPS = 4

LR = 8e-4
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0

EMBED_DIM = 64
DEPTH = 2
NUM_HEADS = 2
MLP_RATIO = 2
DROPOUT = 0.1

FINE_PATCH = 16
FINE_STRIDE = 8

COARSE_PATCH = 32
COARSE_STRIDE = 16

NUM_WORKERS = 0
USE_GRADIENT_CHECKPOINTING = True

ABLATION_DIR = "/kaggle/working/csi_edgevit_ablations"
os.makedirs(ABLATION_DIR, exist_ok=True)


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def model_size_mb(model: nn.Module, model_name: str) -> float:
    temp_path = os.path.join(ABLATION_DIR, f"{model_name}_temp.pth")

    torch.save(model.state_dict(), temp_path)
    size_mb = os.path.getsize(temp_path) / (1024 ** 2)
    os.remove(temp_path)

    return size_mb


def make_head(in_dim: int) -> nn.Sequential:
    return nn.Sequential(
        nn.LayerNorm(in_dim),
        nn.Linear(in_dim, in_dim),
        nn.GELU(),
        nn.Dropout(DROPOUT),
        nn.Linear(in_dim, NUM_CLASSES),
    )


# ============================================================
# PATCH EMBEDDING
# ============================================================

class TemporalPatchEmbedding(nn.Module):
    def __init__(
        self,
        in_channels: int,
        embed_dim: int = 64,
        patch_size: int = 16,
        stride: int = 8,
    ):
        super().__init__()

        self.proj = nn.Conv1d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=stride,
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        x = x.transpose(1, 2)
        x = self.norm(x)

        return x


# ============================================================
# LIGHTWEIGHT MULTI-SCALE TEMPORAL ENCODER BLOCK
# ============================================================

class LiteMultiScaleTEAB(nn.Module):
    def __init__(
        self,
        dim: int = 64,
        num_heads: int = 2,
        mlp_ratio: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.dwconv3 = nn.Conv1d(
            dim,
            dim,
            kernel_size=3,
            padding=1,
            groups=dim,
        )

        self.dwconv5 = nn.Conv1d(
            dim,
            dim,
            kernel_size=5,
            padding=2,
            groups=dim,
        )

        self.dwconv7 = nn.Conv1d(
            dim,
            dim,
            kernel_size=7,
            padding=3,
            groups=dim,
        )

        self.pwconv = nn.Conv1d(
            dim,
            dim,
            kernel_size=1,
        )

        self.bn = nn.BatchNorm1d(dim)

        self.norm1 = nn.LayerNorm(dim)

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = dim * mlp_ratio

        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        residual = x

        x_conv = x.transpose(1, 2)

        x_conv = (
            self.dwconv3(x_conv)
            + self.dwconv5(x_conv)
            + self.dwconv7(x_conv)
        ) / 3.0

        x_conv = self.pwconv(x_conv)
        x_conv = self.bn(x_conv)
        x_conv = F.gelu(x_conv)
        x_conv = x_conv.transpose(1, 2)

        x = residual + x_conv

        residual = x
        x_norm = self.norm1(x)

        attn_out, _ = self.attn(
            x_norm,
            x_norm,
            x_norm,
            need_weights=False,
        )

        x = residual + attn_out

        residual = x
        x_norm = self.norm2(x)

        mlp_out = self.mlp(x_norm)
        gate = self.gate(mlp_out)

        x = residual + gate * mlp_out

        return x


# ============================================================
# CSI EDGEVIT BACKBONE
# ============================================================

class CSIEdgeViTBackbone(nn.Module):
    def __init__(
        self,
        csi_channels: int,
        num_subcarriers: int,
        embed_dim: int = 64,
        depth: int = 2,
        num_heads: int = 2,
        mlp_ratio: int = 2,
        patch_size: int = 16,
        stride: int = 8,
        dropout: float = 0.1,
        use_checkpoint: bool = True,
    ):
        super().__init__()

        self.input_dim = csi_channels * num_subcarriers
        self.use_checkpoint = use_checkpoint

        self.patch_embed = TemporalPatchEmbedding(
            in_channels=self.input_dim,
            embed_dim=embed_dim,
            patch_size=patch_size,
            stride=stride,
        )

        self.blocks = nn.ModuleList(
            [
                LiteMultiScaleTEAB(
                    dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                )
                for _ in range(depth)
            ]
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        batch_size, channels, subcarriers, time_steps = x.shape

        x = x.reshape(
            batch_size,
            channels * subcarriers,
            time_steps,
        )

        x = self.patch_embed(x)

        for block in self.blocks:
            if self.use_checkpoint and self.training:
                x = torch_checkpoint(
                    block,
                    x,
                    use_reentrant=False,
                )
            else:
                x = block(x)

        x = self.norm(x)

        return x


# ============================================================
# CROSS-ATTENTION FUSION
# ============================================================

class CrossAttentionFusion(nn.Module):
    def __init__(
        self,
        dim: int = 64,
        num_heads: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.norm_fine = nn.LayerNorm(dim)
        self.norm_coarse = nn.LayerNorm(dim)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm_out = nn.LayerNorm(dim)

    def forward(self, fine_tokens, coarse_tokens):
        q = self.norm_fine(fine_tokens)
        kv = self.norm_coarse(coarse_tokens)

        attn_out, _ = self.cross_attn(
            q,
            kv,
            kv,
            need_weights=False,
        )

        fused_tokens = self.norm_out(fine_tokens + attn_out)

        return fused_tokens


# ============================================================
# ADAPTIVE FUSION GATE
# ============================================================

class AdaptiveFusionGate(nn.Module):
    def __init__(self, dim: int = 64):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, fine_vec, coarse_vec):
        combined = torch.cat([fine_vec, coarse_vec], dim=-1)

        alpha = self.gate(combined)

        fused = alpha * fine_vec + (1.0 - alpha) * coarse_vec

        return fused, alpha


# ============================================================
# ABLATION MODEL DEFINITIONS
# ============================================================

class A0SingleCSIEdgeViT(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=16,
            stride=8,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.head = make_head(EMBED_DIM)

    def forward(self, x):
        tokens = self.backbone(x)
        vec = tokens.mean(dim=1)

        return self.head(vec)


class A1FineOnly(nn.Module):
    def __init__(self):
        super().__init__()

        self.fine_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=FINE_PATCH,
            stride=FINE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.head = make_head(EMBED_DIM)

    def forward(self, x):
        fine_tokens = self.fine_backbone(x)
        fine_vec = fine_tokens.mean(dim=1)

        return self.head(fine_vec)


class A2CoarseOnly(nn.Module):
    def __init__(self):
        super().__init__()

        self.coarse_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=COARSE_PATCH,
            stride=COARSE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.head = make_head(EMBED_DIM)

    def forward(self, x):
        coarse_tokens = self.coarse_backbone(x)
        coarse_vec = coarse_tokens.mean(dim=1)

        return self.head(coarse_vec)


class A3DualConcat(nn.Module):
    def __init__(self):
        super().__init__()

        self.fine_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=FINE_PATCH,
            stride=FINE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.coarse_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=COARSE_PATCH,
            stride=COARSE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.head = make_head(EMBED_DIM * 2)

    def forward(self, x):
        fine_tokens = self.fine_backbone(x)
        coarse_tokens = self.coarse_backbone(x)

        fine_vec = fine_tokens.mean(dim=1)
        coarse_vec = coarse_tokens.mean(dim=1)

        fused_vec = torch.cat([fine_vec, coarse_vec], dim=-1)

        return self.head(fused_vec)


class A4DualAverage(nn.Module):
    def __init__(self):
        super().__init__()

        self.fine_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=FINE_PATCH,
            stride=FINE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.coarse_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=COARSE_PATCH,
            stride=COARSE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.head = make_head(EMBED_DIM)

    def forward(self, x):
        fine_tokens = self.fine_backbone(x)
        coarse_tokens = self.coarse_backbone(x)

        fine_vec = fine_tokens.mean(dim=1)
        coarse_vec = coarse_tokens.mean(dim=1)

        fused_vec = 0.5 * (fine_vec + coarse_vec)

        return self.head(fused_vec)


class A5CrossAttentionOnly(nn.Module):
    def __init__(self):
        super().__init__()

        self.fine_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=FINE_PATCH,
            stride=FINE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.coarse_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=COARSE_PATCH,
            stride=COARSE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.cross_fusion = CrossAttentionFusion(
            dim=EMBED_DIM,
            num_heads=NUM_HEADS,
            dropout=DROPOUT,
        )

        self.head = make_head(EMBED_DIM)

    def forward(self, x):
        fine_tokens = self.fine_backbone(x)
        coarse_tokens = self.coarse_backbone(x)

        fused_tokens = self.cross_fusion(fine_tokens, coarse_tokens)
        fused_vec = fused_tokens.mean(dim=1)

        return self.head(fused_vec)


class A6GateOnly(nn.Module):
    def __init__(self):
        super().__init__()

        self.fine_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=FINE_PATCH,
            stride=FINE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.coarse_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=COARSE_PATCH,
            stride=COARSE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.adaptive_gate = AdaptiveFusionGate(EMBED_DIM)
        self.head = make_head(EMBED_DIM)

    def forward(self, x):
        fine_tokens = self.fine_backbone(x)
        coarse_tokens = self.coarse_backbone(x)

        fine_vec = fine_tokens.mean(dim=1)
        coarse_vec = coarse_tokens.mean(dim=1)

        fused_vec, _ = self.adaptive_gate(fine_vec, coarse_vec)

        return self.head(fused_vec)


class A7FullDualResolution(nn.Module):
    def __init__(self):
        super().__init__()

        self.fine_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=FINE_PATCH,
            stride=FINE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.coarse_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=COARSE_PATCH,
            stride=COARSE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=USE_GRADIENT_CHECKPOINTING,
        )

        self.cross_fusion = CrossAttentionFusion(
            dim=EMBED_DIM,
            num_heads=NUM_HEADS,
            dropout=DROPOUT,
        )

        self.adaptive_gate = AdaptiveFusionGate(EMBED_DIM)
        self.head = make_head(EMBED_DIM)

    def forward(self, x):
        fine_tokens = self.fine_backbone(x)
        coarse_tokens = self.coarse_backbone(x)

        fused_tokens = self.cross_fusion(fine_tokens, coarse_tokens)

        fine_vec = fused_tokens.mean(dim=1)
        coarse_vec = coarse_tokens.mean(dim=1)

        fused_vec, _ = self.adaptive_gate(fine_vec, coarse_vec)

        return self.head(fused_vec)


# ============================================================
# DATALOADER CREATION
# ============================================================

def make_seeded_loaders(seed: int = 42):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        generator=generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
    )

    test_loader = None

    if test_dataset is not None:
        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            drop_last=False,
            num_workers=NUM_WORKERS,
            pin_memory=DEVICE.type == "cuda",
        )

    return train_loader, val_loader, test_loader


# ============================================================
# LATENCY PROFILING
# ============================================================

@torch.no_grad()
def profile_latency(
    model: nn.Module,
    device: torch.device,
    input_shape: tuple,
    warmup: int = 15,
    runs: int = 40,
):
    model.eval()

    dummy = torch.randn(*input_shape).to(device)

    for _ in range(warmup):
        _ = model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    for _ in range(runs):
        _ = model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    latency_ms = ((end - start) / runs) * 1000

    peak_mem_mb = None

    if device.type == "cuda":
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return latency_ms, peak_mem_mb


# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch_ablation(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    scaler,
    criterion,
):
    model.train()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, desc="Train", leave=False)

    for step, (x, y) in enumerate(progress):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=DEVICE.type == "cuda",
        ):
            logits = model(x)
            loss = criterion(logits, y)
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        should_update = (
            (step + 1) % ACCUM_STEPS == 0
            or (step + 1) == len(loader)
        )

        if should_update:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM,
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * ACCUM_STEPS

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        progress.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

        del x, y, logits, loss, preds

    return {
        "loss": running_loss / len(loader),
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0),
    }


# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_ablation(
    model: nn.Module,
    loader: DataLoader,
    criterion,
    split_name: str = "Val",
):
    model.eval()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    progress = tqdm(loader, desc=split_name, leave=False)

    for x, y in progress:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=DEVICE.type == "cuda",
        ):
            logits = model(x)
            loss = criterion(logits, y)

        running_loss += loss.item()

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, loss, preds

    return {
        "loss": running_loss / len(loader),
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0),
        "labels": labels_all,
        "preds": preds_all,
    }


# ============================================================
# RUN SINGLE ABLATION
# ============================================================

def run_ablation(
    model_name: str,
    model_class,
    seed: int = 42,
):
    print("\n" + "=" * 80)
    print(f"RUNNING: {model_name}")
    print("=" * 80)

    cleanup()
    set_seed(seed)

    train_loader, val_loader, test_loader = make_seeded_loaders(seed)

    model = model_class().to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=DEVICE.type == "cuda",
    )

    best_val_macro_f1 = 0.0
    best_val_acc = 0.0
    best_val_weighted_f1 = 0.0
    best_epoch = 0
    patience_counter = 0

    history = []

    ckpt_path = os.path.join(
        ABLATION_DIR,
        f"{model_name}_best.pth",
    )

    for epoch in range(1, EPOCHS + 1):
        cleanup()

        train_metrics = train_one_epoch_ablation(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            criterion=criterion,
        )

        val_metrics = evaluate_ablation(
            model=model,
            loader=val_loader,
            criterion=criterion,
            split_name=f"Epoch {epoch} Val",
        )

        scheduler.step()

        row = {
            "model": model_name,
            "seed": seed,
            "epoch": epoch,

            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],

            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        }

        history.append(row)

        print(
            f"{model_name} | "
            f"Epoch [{epoch:02d}/{EPOCHS}] | "
            f"Train Acc: {row['train_acc']*100:.2f}% | "
            f"Val Acc: {row['val_acc']*100:.2f}% | "
            f"Val Macro-F1: {row['val_macro_f1']*100:.2f}% | "
            f"Val Weighted-F1: {row['val_weighted_f1']*100:.2f}%"
        )

        improved = row["val_macro_f1"] > best_val_macro_f1 + MIN_DELTA

        if improved:
            best_val_macro_f1 = row["val_macro_f1"]
            best_val_acc = row["val_acc"]
            best_val_weighted_f1 = row["val_weighted_f1"]
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_name": model_name,
                    "seed": seed,
                    "best_epoch": best_epoch,
                    "best_val_acc": best_val_acc,
                    "best_val_macro_f1": best_val_macro_f1,
                    "best_val_weighted_f1": best_val_weighted_f1,
                    "label_mapping": LABEL_MAPPING,
                    "history": history,
                },
                ckpt_path,
            )

            print(
                f"Saved best {model_name} | "
                f"Epoch {best_epoch} | "
                f"Val Acc {best_val_acc*100:.2f}% | "
                f"Val Macro-F1 {best_val_macro_f1*100:.2f}% | "
                f"Val Weighted-F1 {best_val_weighted_f1*100:.2f}%"
            )

        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print(
                f"Early stopping {model_name} at epoch {epoch}. "
                f"Best epoch: {best_epoch}"
            )
            break

    best_ckpt = torch.load(ckpt_path, map_location=DEVICE)

    model.load_state_dict(best_ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    val_final = evaluate_ablation(
        model=model,
        loader=val_loader,
        criterion=criterion,
        split_name=f"{model_name} Final Val",
    )

    test_final = None

    if test_loader is not None:
        test_final = evaluate_ablation(
            model=model,
            loader=test_loader,
            criterion=criterion,
            split_name=f"{model_name} Final Test",
        )

    edge_input_shape = (
        1,
        CSI_CHANNELS,
        TARGET_SUBCARRIERS,
        TARGET_TIME_LEN,
    )

    cuda_latency_ms, peak_mem_mb = profile_latency(
        model=model,
        device=DEVICE,
        input_shape=edge_input_shape,
    )

    model_cpu = model_class().cpu()
    model_cpu.load_state_dict(best_ckpt["model_state_dict"])
    model_cpu.eval()

    cpu_latency_ms, _ = profile_latency(
        model=model_cpu,
        device=torch.device("cpu"),
        input_shape=edge_input_shape,
        warmup=10,
        runs=30,
    )

    params = count_params(model)
    size_mb = model_size_mb(model, model_name)

    summary = {
        "model": model_name,
        "seed": seed,
        "best_epoch": best_epoch,

        "params": params,
        "model_size_mb": size_mb,

        "cuda_latency_ms": cuda_latency_ms,
        "cpu_latency_ms": cpu_latency_ms,
        "peak_mem_mb": peak_mem_mb,

        "val_acc": val_final["accuracy"],
        "val_macro_f1": val_final["macro_f1"],
        "val_weighted_f1": val_final["weighted_f1"],

        "test_acc": test_final["accuracy"] if test_final is not None else None,
        "test_macro_f1": test_final["macro_f1"] if test_final is not None else None,
        "test_weighted_f1": test_final["weighted_f1"] if test_final is not None else None,
    }

    summary_path = os.path.join(
        ABLATION_DIR,
        f"{model_name}_summary.json",
    )

    history_path = os.path.join(
        ABLATION_DIR,
        f"{model_name}_history.csv",
    )

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=4)

    pd.DataFrame(history).to_csv(
        history_path,
        index=False,
    )

    combined_path = os.path.join(
        ABLATION_DIR,
        "ablation_combined_results.csv",
    )

    new_row = pd.DataFrame([summary])

    if os.path.exists(combined_path):
        old_df = pd.read_csv(combined_path)
        old_df = old_df[old_df["model"] != model_name]

        combined_df = pd.concat(
            [old_df, new_row],
            ignore_index=True,
        )
    else:
        combined_df = new_row

    combined_df.to_csv(
        combined_path,
        index=False,
    )

    print("\n" + "-" * 80)
    print(f"{model_name} FINAL SUMMARY")
    print("-" * 80)
    print(json.dumps(summary, indent=4))
    print(f"\nSaved combined results to: {combined_path}")

    cleanup()

    return summary


# ============================================================
# MODEL REGISTRY
# ============================================================

ABLATION_MODELS = {

    "A5_CrossAttentionOnly": A5CrossAttentionOnly

}


print("\nFresh ablation setup complete.")
print("Available models:")

for model_name in ABLATION_MODELS:
    print(" -", model_name)


# ============================================================
# EXAMPLE USAGE
# ============================================================
# Run one ablation:
#
# summary = run_ablation(
#     model_name="A7_FullDualResolution",
#     model_class=A7FullDualResolution,
#     seed=42,
# )
#
# Run all ablations:
#
# all_summaries = []
#
# for model_name, model_class in ABLATION_MODELS.items():
#     summary = run_ablation(
#         model_name=model_name,
#         model_class=model_class,
#         seed=42,
#     )
#     all_summaries.append(summary)
#
# pd.DataFrame(all_summaries).to_csv(
#     os.path.join(ABLATION_DIR, "all_ablation_summaries.csv"),
#     index=False,
# )
# ============================================================

In [ ]:
a0_summary = run_ablation(
    model_name="A0_single_csiedgevit",
    model_class=A0SingleCSIEdgeViT,
    seed=42
)

In [ ]:
a1_summary = run_ablation(
    model_name="A1_fine_only",
    model_class=A1FineOnly,
    seed=42
)

In [ ]:
a2_summary = run_ablation(
    model_name="A2_coarse_only",
    model_class=A2CoarseOnly,
    seed=42
)

In [ ]:
a3_summary = run_ablation(
    model_name="A3_dual_concat",
    model_class=A3DualConcat,
    seed=42
)

In [ ]:
a4_summary = run_ablation(
    model_name="A4_dual_average",
    model_class=A4DualAverage,
    seed=42
)

In [ ]:
a5_summary = run_ablation(
    model_name="A5_cross_attention_only",
    model_class=A5CrossAttentionOnly,
    seed=42
)

In [ ]:
from torch.utils.data import DataLoader

# =========================================================
# CREATE GLOBAL EVAL SPLIT + TEST LOADER FOR ROBUSTNESS
# Run this before robustness code
# =========================================================

EVAL_BATCH_SIZE = 64

if "test_dataset" in globals() and test_dataset is not None:
    EVAL_SPLIT = "test"
    eval_dataset = test_dataset

elif "val_dataset" in globals() and val_dataset is not None:
    EVAL_SPLIT = "val"
    eval_dataset = val_dataset
    print("WARNING: test_dataset not found. Using val_dataset for robustness evaluation.")

elif "train_dataset" in globals() and train_dataset is not None:
    EVAL_SPLIT = "train"
    eval_dataset = train_dataset
    print("WARNING: test_dataset and val_dataset not found. Using train_dataset.")

else:
    raise NameError(
        "No dataset found. You need train_dataset, val_dataset, or test_dataset before running robustness."
    )

test_loader = DataLoader(
    eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS if "NUM_WORKERS" in globals() else 0,
    pin_memory=DEVICE.type == "cuda",
)

print("Created test_loader for robustness.")
print("Evaluation split:", EVAL_SPLIT)
print("Evaluation samples:", len(eval_dataset))

In [ ]:
# =========================================================
# ROBUSTNESS ANALYSIS: CNN vs A5 Cross-Attention CSI-EdgeViT
# Run this AFTER the fixed dataset/eval-split cell
# =========================================================

import os
import gc
import json
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.checkpoint import checkpoint as torch_checkpoint


# =========================================================
# CHECK REQUIRED VARIABLES FROM PREVIOUS CELL
# =========================================================

required_vars = [
    "ROOT",
    "test_loader",
    "test_dataset",
    "EVAL_SPLIT",
    "CSI_CHANNELS",
    "TARGET_SUBCARRIERS",
    "TARGET_TIME_LEN",
    "LABEL_MAPPING",
    "NUM_CLASSES",
    "DEVICE",
]

missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    raise NameError(
        "Missing required variables from previous dataset cell:\n"
        + "\n".join(missing_vars)
    )

print("Using evaluation split:", EVAL_SPLIT)
print("CSI channels:", CSI_CHANNELS)
print("Num classes:", NUM_CLASSES)
print("Label mapping:", LABEL_MAPPING)


# =========================================================
# PATHS
# =========================================================

CNN_CKPT_PATH = "/kaggle/input/models/bickyyadav45/model-cnn/keras/default/1/B1_cnn_1d_best.pth"
A5_CKPT_PATH = "/kaggle/working/csi_edgevit_ablations/A5_cross_attention_only_best.pth"

OUT_DIR = "/kaggle/working/robustness_a5_vs_cnn"
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(CNN_CKPT_PATH):
    raise FileNotFoundError(f"Missing CNN checkpoint: {CNN_CKPT_PATH}")

if not os.path.exists(A5_CKPT_PATH):
    raise FileNotFoundError(
        f"Missing A5 checkpoint: {A5_CKPT_PATH}\n"
        "Run A5 ablation training first, then rerun this robustness cell."
    )


# =========================================================
# CONFIG
# =========================================================

SEED = 42

EMBED_DIM = globals().get("EMBED_DIM", 64)
DEPTH = globals().get("DEPTH", 2)
NUM_HEADS = globals().get("NUM_HEADS", 2)
MLP_RATIO = globals().get("MLP_RATIO", 2)
DROPOUT = globals().get("DROPOUT", 0.1)

FINE_PATCH = 16
FINE_STRIDE = 8

COARSE_PATCH = 32
COARSE_STRIDE = 16

USE_GRADIENT_CHECKPOINTING = False

print("\n========== MODEL CONFIG ==========")
print("EMBED_DIM:", EMBED_DIM)
print("DEPTH:", DEPTH)
print("NUM_HEADS:", NUM_HEADS)
print("MLP_RATIO:", MLP_RATIO)
print("FINE_PATCH:", FINE_PATCH)
print("FINE_STRIDE:", FINE_STRIDE)
print("COARSE_PATCH:", COARSE_PATCH)
print("COARSE_STRIDE:", COARSE_STRIDE)


# =========================================================
# HELPERS
# =========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def get_state_dict(ckpt):
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        return ckpt["model_state_dict"]

    return ckpt


set_seed(SEED)
cleanup()


# =========================================================
# LOAD CHECKPOINTS
# =========================================================

cnn_ckpt = safe_torch_load(CNN_CKPT_PATH, map_location="cpu")
a5_ckpt = safe_torch_load(A5_CKPT_PATH, map_location="cpu")

cnn_state = get_state_dict(cnn_ckpt)
a5_state = get_state_dict(a5_ckpt)

print("\nCheckpoints loaded.")
print("CNN checkpoint:", CNN_CKPT_PATH)
print("A5 checkpoint :", A5_CKPT_PATH)


# =========================================================
# CNN BASELINE MODEL
# =========================================================

class BaselineCNN1D(nn.Module):
    def __init__(self):
        super().__init__()

        input_dim = CSI_CHANNELS * TARGET_SUBCARRIERS

        self.features = nn.Sequential(
            nn.Conv1d(
                input_dim,
                64,
                kernel_size=7,
                stride=2,
                padding=3,
            ),
            nn.BatchNorm1d(64),
            nn.GELU(),

            nn.Conv1d(
                64,
                96,
                kernel_size=5,
                stride=2,
                padding=2,
            ),
            nn.BatchNorm1d(96),
            nn.GELU(),

            nn.Conv1d(
                96,
                128,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
            nn.BatchNorm1d(128),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(1),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(128),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, NUM_CLASSES),
        )

    def forward(self, x):
        batch_size, channels, subcarriers, time_steps = x.shape

        x = x.reshape(
            batch_size,
            channels * subcarriers,
            time_steps,
        )

        x = self.features(x)
        logits = self.head(x)

        return logits


# =========================================================
# A5 CROSS-ATTENTION CSI-EDGEVIT MODEL
# =========================================================

class TemporalPatchEmbedding(nn.Module):
    def __init__(
        self,
        in_channels,
        embed_dim=64,
        patch_size=16,
        stride=8,
    ):
        super().__init__()

        self.proj = nn.Conv1d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=stride,
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        x = x.transpose(1, 2)
        x = self.norm(x)

        return x


class LiteMultiScaleTEAB(nn.Module):
    def __init__(
        self,
        dim=64,
        num_heads=2,
        mlp_ratio=2,
        dropout=0.1,
    ):
        super().__init__()

        self.dwconv3 = nn.Conv1d(
            dim,
            dim,
            kernel_size=3,
            padding=1,
            groups=dim,
        )

        self.dwconv5 = nn.Conv1d(
            dim,
            dim,
            kernel_size=5,
            padding=2,
            groups=dim,
        )

        self.dwconv7 = nn.Conv1d(
            dim,
            dim,
            kernel_size=7,
            padding=3,
            groups=dim,
        )

        self.pwconv = nn.Conv1d(
            dim,
            dim,
            kernel_size=1,
        )

        self.bn = nn.BatchNorm1d(dim)

        self.norm1 = nn.LayerNorm(dim)

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = dim * mlp_ratio

        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        residual = x

        x_conv = x.transpose(1, 2)

        x_conv = (
            self.dwconv3(x_conv)
            + self.dwconv5(x_conv)
            + self.dwconv7(x_conv)
        ) / 3.0

        x_conv = self.pwconv(x_conv)
        x_conv = self.bn(x_conv)
        x_conv = F.gelu(x_conv)
        x_conv = x_conv.transpose(1, 2)

        x = residual + x_conv

        residual = x
        x_norm = self.norm1(x)

        attn_out, _ = self.attn(
            x_norm,
            x_norm,
            x_norm,
            need_weights=False,
        )

        x = residual + attn_out

        residual = x
        x_norm = self.norm2(x)

        mlp_out = self.mlp(x_norm)
        gate = self.gate(mlp_out)

        x = residual + gate * mlp_out

        return x


class CSIEdgeViTBackbone(nn.Module):
    def __init__(
        self,
        csi_channels,
        num_subcarriers,
        embed_dim=64,
        depth=2,
        num_heads=2,
        mlp_ratio=2,
        patch_size=16,
        stride=8,
        dropout=0.1,
        use_checkpoint=False,
    ):
        super().__init__()

        self.input_dim = csi_channels * num_subcarriers
        self.use_checkpoint = use_checkpoint

        self.patch_embed = TemporalPatchEmbedding(
            in_channels=self.input_dim,
            embed_dim=embed_dim,
            patch_size=patch_size,
            stride=stride,
        )

        self.blocks = nn.ModuleList(
            [
                LiteMultiScaleTEAB(
                    dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                )
                for _ in range(depth)
            ]
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        batch_size, channels, subcarriers, time_steps = x.shape

        x = x.reshape(
            batch_size,
            channels * subcarriers,
            time_steps,
        )

        x = self.patch_embed(x)

        for block in self.blocks:
            if self.use_checkpoint and self.training:
                x = torch_checkpoint(
                    block,
                    x,
                    use_reentrant=False,
                )
            else:
                x = block(x)

        x = self.norm(x)

        return x


class CrossAttentionFusion(nn.Module):
    def __init__(
        self,
        dim=64,
        num_heads=2,
        dropout=0.1,
    ):
        super().__init__()

        self.norm_fine = nn.LayerNorm(dim)
        self.norm_coarse = nn.LayerNorm(dim)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm_out = nn.LayerNorm(dim)

    def forward(self, fine_tokens, coarse_tokens):
        q = self.norm_fine(fine_tokens)
        kv = self.norm_coarse(coarse_tokens)

        attn_out, _ = self.cross_attn(
            q,
            kv,
            kv,
            need_weights=False,
        )

        fused_tokens = self.norm_out(fine_tokens + attn_out)

        return fused_tokens


class A5CrossAttentionOnly(nn.Module):
    def __init__(self):
        super().__init__()

        self.fine_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=FINE_PATCH,
            stride=FINE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=False,
        )

        self.coarse_backbone = CSIEdgeViTBackbone(
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            embed_dim=EMBED_DIM,
            depth=DEPTH,
            num_heads=NUM_HEADS,
            mlp_ratio=MLP_RATIO,
            patch_size=COARSE_PATCH,
            stride=COARSE_STRIDE,
            dropout=DROPOUT,
            use_checkpoint=False,
        )

        self.cross_fusion = CrossAttentionFusion(
            dim=EMBED_DIM,
            num_heads=NUM_HEADS,
            dropout=DROPOUT,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(EMBED_DIM),
            nn.Linear(EMBED_DIM, EMBED_DIM),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(EMBED_DIM, NUM_CLASSES),
        )

    def forward(self, x):
        fine_tokens = self.fine_backbone(x)
        coarse_tokens = self.coarse_backbone(x)

        fused_tokens = self.cross_fusion(
            fine_tokens,
            coarse_tokens,
        )

        fused_vec = fused_tokens.mean(dim=1)

        logits = self.head(fused_vec)

        return logits


# =========================================================
# LOAD MODELS
# =========================================================

cnn_model = BaselineCNN1D().to(DEVICE)

try:
    cnn_model.load_state_dict(cnn_state, strict=True)
except RuntimeError as e:
    raise RuntimeError(
        "CNN checkpoint could not be loaded strictly. "
        "This usually means the CNN architecture here does not match the checkpoint.\n\n"
        f"Original error:\n{e}"
    )

cnn_model.eval()

a5_model = A5CrossAttentionOnly().to(DEVICE)

try:
    a5_model.load_state_dict(a5_state, strict=True)
except RuntimeError as e:
    raise RuntimeError(
        "A5 checkpoint could not be loaded strictly. "
        "Check that FINE_PATCH, FINE_STRIDE, COARSE_PATCH, COARSE_STRIDE, "
        "EMBED_DIM, DEPTH, and NUM_HEADS match the training run.\n\n"
        f"Original error:\n{e}"
    )

a5_model.eval()

print("\nCNN and A5 checkpoints loaded successfully.")


# =========================================================
# PERTURBATION FUNCTIONS
# =========================================================

def perturb_clean(x):
    return x


def perturb_gaussian_noise(x, std=0.1):
    return x + torch.randn_like(x) * std


def perturb_random_subcarrier_mask(x, drop_prob=0.3):
    batch_size, channels, subcarriers, time_steps = x.shape

    mask = (
        torch.rand(
            batch_size,
            1,
            subcarriers,
            1,
            device=x.device,
        )
        > drop_prob
    ).float()

    return x * mask


def perturb_contiguous_subcarrier_mask(x, drop_ratio=0.3):
    batch_size, channels, subcarriers, time_steps = x.shape

    width = max(1, int(subcarriers * drop_ratio))
    out = x.clone()

    for b in range(batch_size):
        start = torch.randint(
            low=0,
            high=subcarriers - width + 1,
            size=(1,),
            device=x.device,
        ).item()

        out[b, :, start:start + width, :] = 0.0

    return out


def perturb_temporal_mask(x, drop_ratio=0.2):
    batch_size, channels, subcarriers, time_steps = x.shape

    width = max(1, int(time_steps * drop_ratio))
    out = x.clone()

    for b in range(batch_size):
        start = torch.randint(
            low=0,
            high=time_steps - width + 1,
            size=(1,),
            device=x.device,
        ).item()

        out[b, :, :, start:start + width] = 0.0

    return out


def perturb_time_shift(x, shift=50):
    return torch.roll(
        x,
        shifts=shift,
        dims=-1,
    )


def perturb_crop_resize(x, keep_ratio=0.75):
    batch_size, channels, subcarriers, time_steps = x.shape

    keep_len = max(8, int(time_steps * keep_ratio))
    out_list = []

    for b in range(batch_size):
        start = torch.randint(
            low=0,
            high=time_steps - keep_len + 1,
            size=(1,),
            device=x.device,
        ).item()

        crop = x[b:b + 1, :, :, start:start + keep_len]
        crop = crop.reshape(1, channels * subcarriers, keep_len)

        resized = F.interpolate(
            crop,
            size=time_steps,
            mode="linear",
            align_corners=False,
        )

        resized = resized.reshape(
            1,
            channels,
            subcarriers,
            time_steps,
        )

        out_list.append(resized)

    return torch.cat(out_list, dim=0)


def perturb_amplitude_scale(x, scale=0.7):
    return x * scale


def perturb_combined_mild(x):
    x = perturb_gaussian_noise(x, std=0.10)
    x = perturb_random_subcarrier_mask(x, drop_prob=0.20)
    x = perturb_temporal_mask(x, drop_ratio=0.10)

    return x


def perturb_combined_harsh(x):
    x = perturb_gaussian_noise(x, std=0.20)
    x = perturb_random_subcarrier_mask(x, drop_prob=0.40)
    x = perturb_temporal_mask(x, drop_ratio=0.20)
    x = perturb_crop_resize(x, keep_ratio=0.75)

    return x


# =========================================================
# ROBUSTNESS CONDITIONS
# =========================================================

ROBUSTNESS_CONDITIONS = [
    {
        "name": "clean",
        "fn": perturb_clean,
        "kwargs": {},
        "trials": 1,
    },

    {
        "name": "gaussian_noise_0.05",
        "fn": perturb_gaussian_noise,
        "kwargs": {"std": 0.05},
        "trials": 3,
    },
    {
        "name": "gaussian_noise_0.10",
        "fn": perturb_gaussian_noise,
        "kwargs": {"std": 0.10},
        "trials": 3,
    },
    {
        "name": "gaussian_noise_0.20",
        "fn": perturb_gaussian_noise,
        "kwargs": {"std": 0.20},
        "trials": 3,
    },

    {
        "name": "random_subcarrier_mask_10",
        "fn": perturb_random_subcarrier_mask,
        "kwargs": {"drop_prob": 0.10},
        "trials": 3,
    },
    {
        "name": "random_subcarrier_mask_30",
        "fn": perturb_random_subcarrier_mask,
        "kwargs": {"drop_prob": 0.30},
        "trials": 3,
    },
    {
        "name": "random_subcarrier_mask_50",
        "fn": perturb_random_subcarrier_mask,
        "kwargs": {"drop_prob": 0.50},
        "trials": 3,
    },

    {
        "name": "contiguous_subcarrier_mask_10",
        "fn": perturb_contiguous_subcarrier_mask,
        "kwargs": {"drop_ratio": 0.10},
        "trials": 3,
    },
    {
        "name": "contiguous_subcarrier_mask_30",
        "fn": perturb_contiguous_subcarrier_mask,
        "kwargs": {"drop_ratio": 0.30},
        "trials": 3,
    },

    {
        "name": "temporal_mask_10",
        "fn": perturb_temporal_mask,
        "kwargs": {"drop_ratio": 0.10},
        "trials": 3,
    },
    {
        "name": "temporal_mask_30",
        "fn": perturb_temporal_mask,
        "kwargs": {"drop_ratio": 0.30},
        "trials": 3,
    },

    {
        "name": "time_shift_25",
        "fn": perturb_time_shift,
        "kwargs": {"shift": 25},
        "trials": 1,
    },
    {
        "name": "time_shift_50",
        "fn": perturb_time_shift,
        "kwargs": {"shift": 50},
        "trials": 1,
    },

    {
        "name": "crop_resize_75",
        "fn": perturb_crop_resize,
        "kwargs": {"keep_ratio": 0.75},
        "trials": 3,
    },
    {
        "name": "crop_resize_50",
        "fn": perturb_crop_resize,
        "kwargs": {"keep_ratio": 0.50},
        "trials": 3,
    },

    {
        "name": "amplitude_scale_0.70",
        "fn": perturb_amplitude_scale,
        "kwargs": {"scale": 0.70},
        "trials": 1,
    },
    {
        "name": "amplitude_scale_1.30",
        "fn": perturb_amplitude_scale,
        "kwargs": {"scale": 1.30},
        "trials": 1,
    },

    {
        "name": "combined_mild",
        "fn": perturb_combined_mild,
        "kwargs": {},
        "trials": 3,
    },
    {
        "name": "combined_harsh",
        "fn": perturb_combined_harsh,
        "kwargs": {},
        "trials": 3,
    },
]


# =========================================================
# EVALUATION FUNCTIONS
# =========================================================

@torch.no_grad()
def evaluate_model_under_condition(
    model,
    loader,
    condition,
    trial_seed=42,
):
    set_seed(trial_seed)

    model.eval()
    criterion = nn.CrossEntropyLoss()

    losses = []
    preds_all = []
    labels_all = []

    fn = condition["fn"]
    kwargs = condition["kwargs"]

    progress = tqdm(
        loader,
        desc=condition["name"],
        leave=False,
    )

    for x, y in progress:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        x = fn(x, **kwargs)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=DEVICE.type == "cuda",
        ):
            logits = model(x)
            loss = criterion(logits, y)

        preds = logits.argmax(dim=1)

        losses.append(loss.item())
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, loss, preds

    accuracy = accuracy_score(labels_all, preds_all)
    macro_f1 = f1_score(
        labels_all,
        preds_all,
        average="macro",
        zero_division=0,
    )
    weighted_f1 = f1_score(
        labels_all,
        preds_all,
        average="weighted",
        zero_division=0,
    )

    return {
        "loss": float(np.mean(losses)),
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "labels": labels_all,
        "preds": preds_all,
    }


def evaluate_trials(
    model,
    model_name,
    loader,
    condition,
):
    trial_results = []

    for trial_idx in range(condition["trials"]):
        trial_seed = SEED + 1000 * trial_idx

        result = evaluate_model_under_condition(
            model=model,
            loader=loader,
            condition=condition,
            trial_seed=trial_seed,
        )

        trial_results.append(result)

    losses = np.array([r["loss"] for r in trial_results])
    accs = np.array([r["accuracy"] for r in trial_results])
    macro_f1s = np.array([r["macro_f1"] for r in trial_results])
    weighted_f1s = np.array([r["weighted_f1"] for r in trial_results])

    return {
        "model": model_name,
        "condition": condition["name"],
        "trials": condition["trials"],

        "loss_mean": float(losses.mean()),
        "loss_std": float(losses.std(ddof=1)) if len(losses) > 1 else 0.0,

        "acc_mean": float(accs.mean()),
        "acc_std": float(accs.std(ddof=1)) if len(accs) > 1 else 0.0,

        "macro_f1_mean": float(macro_f1s.mean()),
        "macro_f1_std": float(macro_f1s.std(ddof=1)) if len(macro_f1s) > 1 else 0.0,

        "weighted_f1_mean": float(weighted_f1s.mean()),
        "weighted_f1_std": float(weighted_f1s.std(ddof=1)) if len(weighted_f1s) > 1 else 0.0,
    }


# =========================================================
# RUN ROBUSTNESS ANALYSIS
# =========================================================

models_to_eval = [
    ("CNN", cnn_model),
    ("A5_CrossAttention_CSI_EdgeViT", a5_model),
]

all_rows = []

print("\nStarting robustness evaluation...\n")

for condition in ROBUSTNESS_CONDITIONS:
    print("\n" + "=" * 80)
    print("Condition:", condition["name"])
    print("=" * 80)

    for model_name, model_obj in models_to_eval:
        cleanup()

        row = evaluate_trials(
            model=model_obj,
            model_name=model_name,
            loader=test_loader,
            condition=condition,
        )

        all_rows.append(row)

        print(
            f"{model_name:32s} | "
            f"Acc: {row['acc_mean']*100:.2f} ± {row['acc_std']*100:.2f} | "
            f"Macro-F1: {row['macro_f1_mean']*100:.2f} ± {row['macro_f1_std']*100:.2f} | "
            f"Weighted-F1: {row['weighted_f1_mean']*100:.2f} ± {row['weighted_f1_std']*100:.2f}"
        )

results_df = pd.DataFrame(all_rows)


# =========================================================
# COMPUTE DROPS VS CLEAN
# =========================================================

clean_lookup = {}

for model_name in results_df["model"].unique():
    clean_row = results_df[
        (results_df["model"] == model_name)
        & (results_df["condition"] == "clean")
    ].iloc[0]

    clean_lookup[model_name] = {
        "clean_acc": clean_row["acc_mean"],
        "clean_macro_f1": clean_row["macro_f1_mean"],
        "clean_weighted_f1": clean_row["weighted_f1_mean"],
    }

drop_rows = []

for _, row in results_df.iterrows():
    clean = clean_lookup[row["model"]]

    out = row.to_dict()

    out["acc_drop"] = clean["clean_acc"] - row["acc_mean"]
    out["macro_f1_drop"] = clean["clean_macro_f1"] - row["macro_f1_mean"]
    out["weighted_f1_drop"] = clean["clean_weighted_f1"] - row["weighted_f1_mean"]

    drop_rows.append(out)

results_drop_df = pd.DataFrame(drop_rows)


# =========================================================
# A5 MINUS CNN COMPARISON
# =========================================================

pivot_weighted = results_drop_df.pivot(
    index="condition",
    columns="model",
    values="weighted_f1_mean",
).reset_index()

pivot_weighted["A5_minus_CNN_weighted_f1"] = (
    pivot_weighted["A5_CrossAttention_CSI_EdgeViT"]
    - pivot_weighted["CNN"]
)

pivot_acc = results_drop_df.pivot(
    index="condition",
    columns="model",
    values="acc_mean",
).reset_index()

pivot_acc["A5_minus_CNN_acc"] = (
    pivot_acc["A5_CrossAttention_CSI_EdgeViT"]
    - pivot_acc["CNN"]
)

pivot_macro = results_drop_df.pivot(
    index="condition",
    columns="model",
    values="macro_f1_mean",
).reset_index()

pivot_macro["A5_minus_CNN_macro_f1"] = (
    pivot_macro["A5_CrossAttention_CSI_EdgeViT"]
    - pivot_macro["CNN"]
)

comparison_df = (
    pivot_weighted
    .merge(
        pivot_acc[["condition", "A5_minus_CNN_acc"]],
        on="condition",
    )
    .merge(
        pivot_macro[["condition", "A5_minus_CNN_macro_f1"]],
        on="condition",
    )
)


# =========================================================
# CONVERT TO PERCENTAGES FOR DISPLAY
# =========================================================

display_df = results_drop_df.copy()

percent_cols = [
    "acc_mean",
    "acc_std",
    "macro_f1_mean",
    "macro_f1_std",
    "weighted_f1_mean",
    "weighted_f1_std",
    "acc_drop",
    "macro_f1_drop",
    "weighted_f1_drop",
]

for col in percent_cols:
    display_df[col] = display_df[col] * 100

comparison_display = comparison_df.copy()

comparison_percent_cols = [
    "CNN",
    "A5_CrossAttention_CSI_EdgeViT",
    "A5_minus_CNN_weighted_f1",
    "A5_minus_CNN_acc",
    "A5_minus_CNN_macro_f1",
]

for col in comparison_percent_cols:
    comparison_display[col] = comparison_display[col] * 100


# =========================================================
# SAVE RESULTS
# =========================================================

raw_path = os.path.join(
    OUT_DIR,
    f"robustness_raw_results_{EVAL_SPLIT}_a5_vs_cnn.csv",
)

drop_path = os.path.join(
    OUT_DIR,
    f"robustness_results_with_drops_percent_{EVAL_SPLIT}_a5_vs_cnn.csv",
)

comparison_path = os.path.join(
    OUT_DIR,
    f"robustness_a5_minus_cnn_percent_{EVAL_SPLIT}.csv",
)

results_df.to_csv(raw_path, index=False)
display_df.to_csv(drop_path, index=False)
comparison_display.to_csv(comparison_path, index=False)

print("\n========== SAVED FILES ==========")
print(raw_path)
print(drop_path)
print(comparison_path)


# =========================================================
# DISPLAY RESULTS
# =========================================================

print("\n========== ROBUSTNESS RESULTS WITH DROPS (%) ==========")
display(display_df)

print("\n========== A5 MINUS CNN COMPARISON (%) ==========")
display(comparison_display)

wins = comparison_display[
    comparison_display["A5_minus_CNN_weighted_f1"] > 0
].copy()

print("\n========== CONDITIONS WHERE A5 BEATS CNN ON WEIGHTED-F1 ==========")

if len(wins) == 0:
    print("A5 did not beat CNN on weighted-F1 under the tested perturbations.")
else:
    display(
        wins[
            [
                "condition",
                "CNN",
                "A5_CrossAttention_CSI_EdgeViT",
                "A5_minus_CNN_weighted_f1",
                "A5_minus_CNN_acc",
                "A5_minus_CNN_macro_f1",
            ]
        ]
    )

cleanup()

In [ ]:
a6_summary = run_ablation(
    model_name="A6_gate_only",
    model_class=A6GateOnly, 
    seed=42
)

In [ ]:
a7_summary = run_ablation(
    model_name="A7_full_dual_resolution",
    model_class=A7FullDualResolution,
    seed=42
)

In [ ]:
# =========================================================
# CELL 9: DISPLAY FINAL ABLATION TABLE
# =========================================================

combined_path = "/kaggle/working/ablation_results/ablation_combined_results.csv"

ablation_df = pd.read_csv(combined_path)

order = [
    "A0_single_csiedgevit",
    "A1_fine_only",
    "A2_coarse_only",
    "A3_dual_concat",
    "A4_dual_average",
    "A5_cross_attention_only",
    "A6_gate_only",
    "A7_full_dual_resolution"
]

ablation_df["order"] = ablation_df["model"].apply(
    lambda x: order.index(x) if x in order else 999
)

ablation_df = ablation_df.sort_values("order").drop(columns=["order"])

display_cols = [
    "model",
    "best_epoch",
    "params",
    "model_size_mb",
    "cpu_latency_ms",
    "cuda_latency_ms",
    "peak_mem_mb",
    "val_acc",
    "val_macro_f1",
    "val_weighted_f1",
    "test_acc",
    "test_macro_f1",
    "test_weighted_f1"
]

display_df = ablation_df[display_cols].copy()

for col in [
    "val_acc",
    "val_macro_f1",
    "val_weighted_f1",
    "test_acc",
    "test_macro_f1",
    "test_weighted_f1"
]:
    display_df[col] = display_df[col] * 100

display(display_df)

output_path = "/kaggle/working/ablation_results/ablation_table_percent.csv"
display_df.to_csv(output_path, index=False)

print("Saved final ablation table:")
print(output_path)

In [ ]:
# =========================================================
# BASELINE FRAMEWORK: CNN, LSTM, BiLSTM ONLY
# =========================================================

import os
import gc
import time
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# =========================================================
# CHECK REQUIRED VARIABLES
# =========================================================

required_names = [
    "train_dataset", "val_dataset", "test_dataset",
    "DEVICE", "BATCH_SIZE", "NUM_WORKERS",
    "ACCUM_STEPS", "LR", "WEIGHT_DECAY",
    "EPOCHS", "PATIENCE", "MIN_DELTA",
    "MAX_GRAD_NORM", "TARGET_SUBCARRIERS",
    "TARGET_TIME_LEN", "CSI_CHANNELS", "NUM_CLASSES",
    "LABEL_MAPPING", "INV_LABEL_MAPPING", "DROPOUT"
]

missing = [name for name in required_names if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing required variables from setup cell:\n"
        + "\n".join(missing)
        + "\n\nRun the fresh dataset/setup cell first."
    )

BASELINE_DIR = "/kaggle/working/baseline_results"
os.makedirs(BASELINE_DIR, exist_ok=True)

BASELINE_EPOCHS = EPOCHS
BASELINE_PATIENCE = PATIENCE
BASELINE_MIN_DELTA = MIN_DELTA

# Optional MACs/FLOPs
try:
    from thop import profile
    THOP_AVAILABLE = True
    print("✅ thop available. MACs will be computed.")
except Exception:
    THOP_AVAILABLE = False
    print("⚠️ thop not available. MACs skipped. Optional: !pip install -q thop")

# =========================================================
# HELPERS
# =========================================================

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

def make_baseline_loaders(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        generator=g
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda"
    )

    test_loader = None
    if test_dataset is not None:
        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            drop_last=False,
            num_workers=NUM_WORKERS,
            pin_memory=DEVICE.type == "cuda"
        )

    return train_loader, val_loader, test_loader

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def model_size_mb(model, model_name):
    temp_path = os.path.join(BASELINE_DIR, f"{model_name}_temp_size.pth")
    torch.save(model.state_dict(), temp_path)
    size_mb = os.path.getsize(temp_path) / (1024 ** 2)
    os.remove(temp_path)
    return size_mb

@torch.no_grad()
def profile_latency(model, device, input_shape, warmup=15, runs=40):
    model.eval()
    dummy = torch.randn(*input_shape).to(device)

    for _ in range(warmup):
        _ = model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    for _ in range(runs):
        _ = model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    latency_ms = ((end - start) / runs) * 1000

    peak_mem_mb = None
    if device.type == "cuda":
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return latency_ms, peak_mem_mb

def compute_macs(model_class, state_dict, input_shape):
    if not THOP_AVAILABLE:
        return None, None

    try:
        model_cpu = model_class().cpu().eval()
        model_cpu.load_state_dict(state_dict)

        dummy = torch.randn(*input_shape).cpu()

        macs, profiled_params = profile(
            model_cpu,
            inputs=(dummy,),
            verbose=False
        )

        return float(macs), float(profiled_params)

    except Exception as e:
        print("⚠️ MAC profiling failed:", e)
        return None, None

# =========================================================
# MODEL 1: CNN
# =========================================================

class BaselineCNN1D(nn.Module):
    def __init__(self):
        super().__init__()

        input_dim = CSI_CHANNELS * TARGET_SUBCARRIERS

        self.features = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),

            nn.Conv1d(64, 96, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(96),
            nn.GELU(),

            nn.Conv1d(96, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(128),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, NUM_CLASSES)
        )

    def forward(self, x):
        B, C, K, T = x.shape
        x = x.reshape(B, C * K, T)
        x = self.features(x)
        return self.head(x)

# =========================================================
# MODEL 2: LSTM
# =========================================================

class BaselineLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        input_dim = CSI_CHANNELS * TARGET_SUBCARRIERS
        hidden_dim = 96

        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU()
        )

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=DROPOUT,
            bidirectional=False
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, NUM_CLASSES)
        )

    def forward(self, x):
        B, C, K, T = x.shape
        x = x.reshape(B, C * K, T).transpose(1, 2)  # (B, T, C*K)

        x = self.input_proj(x)
        out, _ = self.lstm(x)

        vec = out.mean(dim=1)
        return self.head(vec)

# =========================================================
# MODEL 3: BiLSTM
# =========================================================

class BaselineBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        input_dim = CSI_CHANNELS * TARGET_SUBCARRIERS
        hidden_dim = 64

        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU()
        )

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=DROPOUT,
            bidirectional=True
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim * 2, NUM_CLASSES)
        )

    def forward(self, x):
        B, C, K, T = x.shape
        x = x.reshape(B, C * K, T).transpose(1, 2)

        x = self.input_proj(x)
        out, _ = self.lstm(x)

        vec = out.mean(dim=1)
        return self.head(vec)

# =========================================================
# TRAIN / EVALUATE
# =========================================================

def train_one_epoch_baseline(model, loader, optimizer, scaler, criterion):
    model.train()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    optimizer.zero_grad(set_to_none=True)

    for step, (x, y) in enumerate(tqdm(loader, desc="Train", leave=False)):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(x)
            loss = criterion(logits, y)
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * ACCUM_STEPS

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, loss, preds

    return {
        "loss": running_loss / len(loader),
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro"),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted")
    }

@torch.no_grad()
def evaluate_baseline(model, loader, criterion, split_name="Val"):
    model.eval()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    for x, y in tqdm(loader, desc=split_name, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(x)
            loss = criterion(logits, y)

        running_loss += loss.item()

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, loss, preds

    return {
        "loss": running_loss / len(loader),
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro"),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted"),
        "labels": labels_all,
        "preds": preds_all
    }

def run_baseline(model_name, model_class, seed=42):
    print("\n" + "=" * 80)
    print(f"RUNNING BASELINE: {model_name}")
    print("=" * 80)

    cleanup()
    set_seed(seed)

    train_loader, val_loader, test_loader = make_baseline_loaders(seed)

    model = model_class().to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=BASELINE_EPOCHS
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda")
    )

    best_val_macro_f1 = 0.0
    best_val_acc = 0.0
    best_val_weighted_f1 = 0.0
    best_epoch = 0
    patience_counter = 0

    history = []
    ckpt_path = os.path.join(BASELINE_DIR, f"{model_name}_best.pth")

    for epoch in range(1, BASELINE_EPOCHS + 1):
        cleanup()

        train_metrics = train_one_epoch_baseline(
            model,
            train_loader,
            optimizer,
            scaler,
            criterion
        )

        val_metrics = evaluate_baseline(
            model,
            val_loader,
            criterion,
            split_name=f"Epoch {epoch} Val"
        )

        scheduler.step()

        row = {
            "model": model_name,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"]
        }

        history.append(row)

        print(
            f"{model_name} | Epoch [{epoch:02d}/{BASELINE_EPOCHS}] | "
            f"Train Acc: {row['train_acc']*100:.2f}% | "
            f"Val Acc: {row['val_acc']*100:.2f}% | "
            f"Val Macro-F1: {row['val_macro_f1']*100:.2f}% | "
            f"Val Weighted-F1: {row['val_weighted_f1']*100:.2f}%"
        )

        if row["val_macro_f1"] > best_val_macro_f1 + BASELINE_MIN_DELTA:
            best_val_macro_f1 = row["val_macro_f1"]
            best_val_acc = row["val_acc"]
            best_val_weighted_f1 = row["val_weighted_f1"]
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_name": model_name,
                    "seed": seed,
                    "best_epoch": best_epoch,
                    "best_val_acc": best_val_acc,
                    "best_val_macro_f1": best_val_macro_f1,
                    "best_val_weighted_f1": best_val_weighted_f1,
                    "label_mapping": LABEL_MAPPING,
                    "history": history
                },
                ckpt_path
            )

            print(
                f"✅ Saved best {model_name} | "
                f"Epoch {best_epoch} | "
                f"Val Acc {best_val_acc*100:.2f}% | "
                f"Val Macro-F1 {best_val_macro_f1*100:.2f}% | "
                f"Val Weighted-F1 {best_val_weighted_f1*100:.2f}%"
            )

        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{BASELINE_PATIENCE}")

        if patience_counter >= BASELINE_PATIENCE:
            print(f"Early stopping {model_name} at epoch {epoch}. Best epoch: {best_epoch}")
            break

    # Load best checkpoint
    best_ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    val_final = evaluate_baseline(
        model,
        val_loader,
        criterion,
        split_name=f"{model_name} Final Val"
    )

    test_final = None

    if test_loader is not None:
        test_final = evaluate_baseline(
            model,
            test_loader,
            criterion,
            split_name=f"{model_name} Final Test"
        )

    target_names = [INV_LABEL_MAPPING[i] for i in range(NUM_CLASSES)]

    val_report = classification_report(
        val_final["labels"],
        val_final["preds"],
        target_names=target_names,
        digits=4
    )

    val_cm = confusion_matrix(
        val_final["labels"],
        val_final["preds"]
    )

    with open(os.path.join(BASELINE_DIR, f"{model_name}_val_report.txt"), "w") as f:
        f.write(val_report)

    pd.DataFrame(
        val_cm,
        index=target_names,
        columns=target_names
    ).to_csv(os.path.join(BASELINE_DIR, f"{model_name}_val_confusion_matrix.csv"))

    if test_final is not None:
        test_report = classification_report(
            test_final["labels"],
            test_final["preds"],
            target_names=target_names,
            digits=4
        )

        test_cm = confusion_matrix(
            test_final["labels"],
            test_final["preds"]
        )

        with open(os.path.join(BASELINE_DIR, f"{model_name}_test_report.txt"), "w") as f:
            f.write(test_report)

        pd.DataFrame(
            test_cm,
            index=target_names,
            columns=target_names
        ).to_csv(os.path.join(BASELINE_DIR, f"{model_name}_test_confusion_matrix.csv"))

    edge_input_shape = (1, CSI_CHANNELS, TARGET_SUBCARRIERS, TARGET_TIME_LEN)

    cuda_latency_ms, peak_mem_mb = profile_latency(
        model,
        DEVICE,
        edge_input_shape
    )

    model_cpu = model_class().cpu()
    model_cpu.load_state_dict(best_ckpt["model_state_dict"])
    model_cpu.eval()

    cpu_latency_ms, _ = profile_latency(
        model_cpu,
        torch.device("cpu"),
        edge_input_shape,
        warmup=10,
        runs=30
    )

    params = count_params(model)
    size_mb = model_size_mb(model, model_name)

    macs, profiled_params = compute_macs(
        model_class,
        best_ckpt["model_state_dict"],
        edge_input_shape
    )

    summary = {
        "model": model_name,
        "seed": seed,
        "best_epoch": best_epoch,
        "params": params,
        "profiled_params": profiled_params,
        "model_size_mb": size_mb,
        "macs": macs,
        "macs_million": macs / 1e6 if macs is not None else None,
        "cuda_latency_ms": cuda_latency_ms,
        "cpu_latency_ms": cpu_latency_ms,
        "peak_mem_mb": peak_mem_mb,
        "val_acc": val_final["accuracy"],
        "val_macro_f1": val_final["macro_f1"],
        "val_weighted_f1": val_final["weighted_f1"],
        "test_acc": test_final["accuracy"] if test_final is not None else None,
        "test_macro_f1": test_final["macro_f1"] if test_final is not None else None,
        "test_weighted_f1": test_final["weighted_f1"] if test_final is not None else None
    }

    with open(os.path.join(BASELINE_DIR, f"{model_name}_summary.json"), "w") as f:
        json.dump(summary, f, indent=4)

    pd.DataFrame(history).to_csv(
        os.path.join(BASELINE_DIR, f"{model_name}_history.csv"),
        index=False
    )

    combined_path = os.path.join(BASELINE_DIR, "baseline_cnn_lstm_bilstm_results.csv")
    new_row = pd.DataFrame([summary])

    if os.path.exists(combined_path):
        old_df = pd.read_csv(combined_path)
        old_df = old_df[old_df["model"] != model_name]
        combined_df = pd.concat([old_df, new_row], ignore_index=True)
    else:
        combined_df = new_row

    combined_df.to_csv(combined_path, index=False)

    print("\n" + "-" * 80)
    print(f"{model_name} FINAL SUMMARY")
    print("-" * 80)
    print(json.dumps(summary, indent=4))
    print(f"\nSaved combined results to: {combined_path}")

    cleanup()

    return summary

print("✅ CNN / LSTM / BiLSTM baseline framework ready.")

In [ ]:
cnn_summary = run_baseline(
    model_name="B1_cnn_1d",
    model_class=BaselineCNN1D,
    seed=42
)

In [ ]:
lstm_summary = run_baseline(
    model_name="B2_lstm",
    model_class=BaselineLSTM,
    seed=42
)

In [ ]:
bilstm_summary = run_baseline(
    model_name="B3_bilstm",
    model_class=BaselineBiLSTM,
    seed=42
)

In [ ]:
# =========================================================
# DISPLAY CNN / LSTM / BiLSTM BASELINE TABLE
# =========================================================

baseline_path = "/kaggle/working/baseline_results/baseline_cnn_lstm_bilstm_results.csv"

baseline_df = pd.read_csv(baseline_path)

order = [
    "B1_cnn_1d",
    "B2_lstm",
    "B3_bilstm"
]

baseline_df["order"] = baseline_df["model"].apply(
    lambda x: order.index(x) if x in order else 999
)

baseline_df = baseline_df.sort_values("order").drop(columns=["order"])

display_cols = [
    "model",
    "best_epoch",
    "params",
    "model_size_mb",
    "macs_million",
    "cpu_latency_ms",
    "cuda_latency_ms",
    "peak_mem_mb",
    "val_acc",
    "val_macro_f1",
    "val_weighted_f1",
    "test_acc",
    "test_macro_f1",
    "test_weighted_f1"
]

display_df = baseline_df[display_cols].copy()

for col in [
    "val_acc",
    "val_macro_f1",
    "val_weighted_f1",
    "test_acc",
    "test_macro_f1",
    "test_weighted_f1"
]:
    display_df[col] = display_df[col] * 100

display(display_df)

output_path = "/kaggle/working/baseline_results/cnn_lstm_bilstm_table_percent.csv"
display_df.to_csv(output_path, index=False)

print("Saved table:")
print(output_path)

In [ ]:
# =========================================================
# CELL 5: DEFINE REMAINING BASELINE MODELS
# CNN-LSTM, Transformer Encoder, Temporal ViT, PatchTST-Lite
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# Check that previous baseline framework exists
if "run_baseline" not in globals():
    raise RuntimeError("run_baseline() is not defined. Run the CNN/LSTM/BiLSTM baseline framework cell first.")

# =========================================================
# MODEL 4: CNN-LSTM
# =========================================================

class BaselineCNNLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        input_dim = CSI_CHANNELS * TARGET_SUBCARRIERS
        conv_dim = 96
        hidden_dim = 96

        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),

            nn.Conv1d(64, conv_dim, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(conv_dim),
            nn.GELU()
        )

        self.lstm = nn.LSTM(
            input_size=conv_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=DROPOUT,
            bidirectional=False
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, NUM_CLASSES)
        )

    def forward(self, x):
        # x: (B, C, K, T)
        B, C, K, T = x.shape

        x = x.reshape(B, C * K, T)   # (B, C*K, T)
        x = self.conv(x)             # (B, D, N)
        x = x.transpose(1, 2)        # (B, N, D)

        out, _ = self.lstm(x)
        vec = out.mean(dim=1)

        return self.head(vec)


# =========================================================
# MODEL 5: Transformer Encoder
# =========================================================

class BaselineTransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        input_dim = CSI_CHANNELS * TARGET_SUBCARRIERS
        embed_dim = 64
        patch_size = 16
        stride = 8

        self.patch_embed = nn.Conv1d(
            in_channels=input_dim,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=stride
        )

        num_tokens = ((TARGET_TIME_LEN - patch_size) // stride) + 1

        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=2,
            dim_feedforward=embed_dim * 2,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(embed_dim, NUM_CLASSES)
        )

    def forward(self, x):
        # x: (B, C, K, T)
        B, C, K, T = x.shape

        x = x.reshape(B, C * K, T)   # (B, C*K, T)
        x = self.patch_embed(x)      # (B, D, N)
        x = x.transpose(1, 2)        # (B, N, D)

        N = x.shape[1]
        x = x + self.pos_embed[:, :N, :]

        x = self.encoder(x)
        x = self.norm(x)

        vec = x.mean(dim=1)

        return self.head(vec)


# =========================================================
# MODEL 6: Temporal ViT
# =========================================================

class BaselineTemporalViT(nn.Module):
    def __init__(self):
        super().__init__()

        input_dim = CSI_CHANNELS * TARGET_SUBCARRIERS
        embed_dim = 64
        patch_size = 16
        stride = 16

        self.patch_embed = nn.Conv1d(
            in_channels=input_dim,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=stride
        )

        num_tokens = ((TARGET_TIME_LEN - patch_size) // stride) + 1

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens + 1, embed_dim))

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=2,
            dim_feedforward=embed_dim * 2,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(embed_dim, NUM_CLASSES)
        )

    def forward(self, x):
        # x: (B, C, K, T)
        B, C, K, T = x.shape

        x = x.reshape(B, C * K, T)   # (B, C*K, T)
        x = self.patch_embed(x)      # (B, D, N)
        x = x.transpose(1, 2)        # (B, N, D)

        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_token, x], dim=1)

        N = x.shape[1]
        x = x + self.pos_embed[:, :N, :]

        x = self.encoder(x)
        x = self.norm(x)

        cls_vec = x[:, 0]

        return self.head(cls_vec)


# =========================================================
# MODEL 7: PatchTST-style Lite Baseline
# =========================================================

class BaselinePatchTSTLite(nn.Module):
    def __init__(self):
        super().__init__()

        self.num_vars = CSI_CHANNELS * TARGET_SUBCARRIERS
        self.patch_len = 16
        self.stride = 8
        self.embed_dim = 32

        num_patches = ((TARGET_TIME_LEN - self.patch_len) // self.stride) + 1

        self.patch_proj = nn.Linear(self.patch_len, self.embed_dim)

        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches, self.embed_dim)
        )
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim,
            nhead=2,
            dim_feedforward=self.embed_dim * 2,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.norm = nn.LayerNorm(self.embed_dim)

        self.head = nn.Sequential(
            nn.Linear(self.embed_dim, 64),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x):
        # x: (B, C, K, T)
        B, C, K, T = x.shape

        x = x.reshape(B, C * K, T)  # (B, V, T)

        patches = x.unfold(
            dimension=2,
            size=self.patch_len,
            step=self.stride
        )  # (B, V, N, P)

        B, V, N, P = patches.shape

        patches = patches.reshape(B * V, N, P)

        z = self.patch_proj(patches)  # (B*V, N, D)
        z = z + self.pos_embed[:, :N, :]

        z = self.encoder(z)
        z = self.norm(z)

        z = z.mean(dim=1)             # (B*V, D)
        z = z.reshape(B, V, self.embed_dim)

        vec = z.mean(dim=1)           # aggregate variables

        return self.head(vec)

print("✅ CNN-LSTM, Transformer, Temporal ViT, and PatchTST-Lite models defined.")

In [ ]:
cnnlstm_summary = run_baseline(
    model_name="B4_cnn_lstm",
    model_class=BaselineCNNLSTM,
    seed=42
)

In [ ]:
transformer_summary = run_baseline(
    model_name="B5_transformer_encoder",
    model_class=BaselineTransformerEncoder,
    seed=42
)

In [ ]:
vit_summary = run_baseline(
    model_name="B6_temporal_vit",
    model_class=BaselineTemporalViT,
    seed=42
)

In [ ]:
patchtst_summary = run_baseline(
    model_name="B7_patchtst_lite",
    model_class=BaselinePatchTSTLite,
    seed=42
)

In [ ]:
# =========================================================
# DISPLAY ALL BASELINE RESULTS
# =========================================================

import os
import pandas as pd

baseline_path = "/kaggle/working/baseline_results/baseline_cnn_lstm_bilstm_results.csv"

if not os.path.exists(baseline_path):
    raise FileNotFoundError(f"Could not find baseline result file: {baseline_path}")

baseline_df = pd.read_csv(baseline_path)

order = [
    "B1_cnn_1d",
    "B2_lstm",
    "B3_bilstm",
    "B4_cnn_lstm",
    "B5_transformer_encoder",
    "B6_temporal_vit",
    "B7_patchtst_lite"
]

baseline_df["order"] = baseline_df["model"].apply(
    lambda x: order.index(x) if x in order else 999
)

baseline_df = baseline_df.sort_values("order").drop(columns=["order"])

display_cols = [
    "model",
    "best_epoch",
    "params",
    "model_size_mb",
    "macs_million",
    "cpu_latency_ms",
    "cuda_latency_ms",
    "peak_mem_mb",
    "val_acc",
    "val_macro_f1",
    "val_weighted_f1",
    "test_acc",
    "test_macro_f1",
    "test_weighted_f1"
]

display_df = baseline_df[display_cols].copy()

for col in [
    "val_acc",
    "val_macro_f1",
    "val_weighted_f1",
    "test_acc",
    "test_macro_f1",
    "test_weighted_f1"
]:
    display_df[col] = display_df[col] * 100

display(display_df)

output_path = "/kaggle/working/baseline_results/all_baseline_table_percent.csv"
display_df.to_csv(output_path, index=False)

print("Saved all baseline table:")
print(output_path)